# ERP CORE LRP

Dataset key: `erp_core_lrp_clean`

Source: ERP CORE OSF/GitHub LRP processed response-locked EEGLAB epoch files

This notebook owns the source-specific import/load cell for this dataset. Plotting stays in the shared Week 15 helper module.


In [ ]:
import Pkg

function find_repo_root()
    candidates = unique(normpath.([
        pwd(),
        joinpath(pwd(), ".."),
        joinpath(pwd(), "..", ".."),
        joinpath(pwd(), "..", "..", ".."),
    ]))
    for candidate in candidates
        if isdir(joinpath(candidate, "notebooks")) && isdir(joinpath(candidate, "scripts"))
            return candidate
        end
    end
    error("Could not locate repository root from pwd=$(pwd()).")
end

const REPO_ROOT = find_repo_root()
const NOTEBOOK_DIR = joinpath(REPO_ROOT, "notebooks", "week_19", "data_sources")
const DATASETS_ROOT = joinpath(REPO_ROOT, "notebooks", "datasets")
const WEEK19_DOWNLOADS = joinpath(REPO_ROOT, "notebooks", "week_19", "downloads")
const PYTHON = begin
    venv_python = joinpath(REPO_ROOT, ".venv_8bit", "bin", "python")
    isfile(venv_python) ? venv_python : "python"
end

Pkg.activate(joinpath(REPO_ROOT, "notebooks", "model_test"))

using CairoMakie
using CSV
using DataFrames
using HDF5
using JSON3
using Printf
using Statistics

include(joinpath(REPO_ROOT, "notebooks", "week_15", "try_new_data_helpers.jl"))
using .Week15TryNewData

mkpath(WEEK19_DOWNLOADS)
RNG_SEED = Int(mod(time_ns(), UInt64(typemax(Int))))
println("Repo root: ", REPO_ROOT)
println("Python: ", PYTHON)
println("RNG seed: ", RNG_SEED)


In [ ]:
function bundle_files(dataset_key::AbstractString)
    dir = joinpath(DATASETS_ROOT, dataset_key)
    return (
        dir = dir,
        h5 = joinpath(dir, "epochs.hdf5"),
        events = joinpath(dir, "events.csv"),
        metadata = joinpath(dir, "metadata.json"),
    )
end

function standard_bundle_ready(dataset_key::AbstractString)
    files = bundle_files(dataset_key)
    all(isfile, [files.h5, files.events, files.metadata]) || return false
    return h5open(files.h5, "r") do f
        if haskey(f, "subjects")
            return length(keys(f["subjects"])) > 0
        end
        return true
    end
end

function maybe_run_import!(cmd::Cmd; force::Bool = false)
    if RUN_IMPORT || force
        run(cmd)
    else
        @info "RUN_IMPORT=false; not running importer. Set RUN_IMPORT=true in this notebook to rebuild/download."
    end
end

const DATASET_KEY = "erp_core_lrp_clean"
const SOURCE_NOTE = "ERP CORE OSF/GitHub LRP processed response-locked EEGLAB epoch files"
const RUN_IMPORT = false
const PREPARE_SCRIPT = joinpath(REPO_ROOT, "scripts", "prepare_erp_core_clean_datasets.py")
const IMPORT_ARGS = String[
    "--output-root",
    DATASETS_ROOT,
    "--components",
    "lrp",
    "--subjects",
    "1:4",
]

@assert isfile(PREPARE_SCRIPT) "Missing importer: $PREPARE_SCRIPT"

files = bundle_files(DATASET_KEY)
if RUN_IMPORT
    println("Forcing rebuild: ", files.dir)
    maybe_run_import!(Cmd(vcat([PYTHON, PREPARE_SCRIPT], IMPORT_ARGS)); force = true)
elseif standard_bundle_ready(DATASET_KEY)
    println("Bundle already ready: ", files.dir)
else
    println("Bundle missing/incomplete: ", files.dir)
    maybe_run_import!(Cmd(vcat([PYTHON, PREPARE_SCRIPT], IMPORT_ARGS)))
end


In [ ]:
TARGET_SIZE = nothing
N_SAMPLES_PER_SORT = 16
N_COLS = 4

if standard_bundle_ready(DATASET_KEY)
    bundle = load_clean_dataset_bundle(DATASET_KEY)
    display(external_dataset_summary_df([bundle]))
    display(available_sort_columns_df([bundle]))

    sort_columns = available_sort_columns(bundle)
    sort_audit = sort_order_audit_df(bundle; sort_columns = sort_columns, include_merged = true)
    display(sort_audit)
    @assert all(sort_audit.status .== "ok") "Sort-order audit failed for $(DATASET_KEY)."

    plot_all_dataset_sort_previews(bundle;
        sort_columns = sort_columns,
        n_samples = N_SAMPLES_PER_SORT,
        target_size = TARGET_SIZE,
        rng_seed = RNG_SEED,
        n_cols = N_COLS,
        merge_subjects = true,
        baseline_correct = false,
        prefer_preferred_channels = true,
    )
else
    files = bundle_files(DATASET_KEY)
    println("Bundle unavailable/incomplete: ", files.dir)
    if isdefined(Main, :SOURCE_NOTE)
        println("Source note: ", SOURCE_NOTE)
    end
    println("No preview generated for this notebook until a complete bundle exists.")
end
